# Shopify API Authentication

Index:
- [API Keys and Auth](#API-Keys-and-Auth)
- [Demo and Setup](##initial-setup-using-first-and-last-filters)
- [Bulk Operations](## Bulk Operations)

<!-- Initial Setup Using First and Last Filters -->


### API Keys and Auth

In [1]:
# Import Libraries
import os
import pandas as pd
import requests
import json

# Custom Functions
import src.utils as utils
import src.df_functions as dffx
import src.queries as queries
import src.gcloud as gcloud

# Datetime Packages
from zoneinfo import ZoneInfo
from datetime import datetime as dt
import dateutil.parser as du
import time

# Services Libraries
from shopify_app import ShopifyApp
from google.cloud import bigquery

# Load Shopify Secrets
SHOPIFY_CLIENT_ID = os.getenv("SHOPIFY_CLIENT_ID")
SHOPIFY_SECRET = os.getenv("SHOPIFY_SECRET")
# merchant = os.getenv("MERCHANT")
# version = '2022-04'

# Load Google Cloud services account and BigQuery Client
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'cloud_python_private_key.json'

client = bigquery.Client()

## Connect to Shopify

#### Connect to Shopify to get an access token

In [2]:
SHOPIFY_ACCESS_TOKEN = utils.get_credentials(SHOPIFY_CLIENT_ID,SHOPIFY_SECRET)

## Bulk Operations



#### Bulk Operation Query

In [4]:
bulk_query_response = utils.bulk_query_request(queries.yesterday,SHOPIFY_ACCESS_TOKEN)

In [10]:
# bulk_query_response

#### Getting Bulk Operation Status. 

Eventually, will create a function to query to check status and if complete and without errors, then grab the URL with the data.

In [28]:
# orders = utils.poll_for_result(SHOPIFY_ACCESS_TOKEN)

In [14]:
orders_df_complete = dffx.add_data_to_df(orders)

In [26]:
# orders_df_complete.head()

In [16]:
new_df = dffx.process_data(orders_df_complete)

In [27]:
# new_df.head()

In [18]:
table_id = "glowing-market-295819.shopify.test_table_function_v2"

In [19]:
gcloud.bigquery_write_table(client, new_df, table_id)

/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Loaded 195 rows and 39 columns to glowing-market-295819.shopify.test_table_function_v2


### Last 30 Days Workthrough

In [1]:
import datetime as dt

now = dt.date.today()
strf_time = now - dt.timedelta(days = 7)

In [6]:
today = dt.date.today()
last_30_days = dt.date.strftime(today - dt.timedelta(days = 30),'%Y-%m-%d')

In [7]:
last_30_days

'2026-03-09'

In [2]:
print(strf_time)
string_date = dt.date.strftime(strf_time,'%Y-%m-%d')

2026-04-01


In [3]:
last_30_days = """
        mutation {
  bulkOperationRunQuery(
   query: \"""
    query { 
orders(query: "created_at:>=2026-02-23 created_at:<={string_date}") {
edges {
node {
id
name
email
createdAt
cancellation {
    staffNote
    } # closes cancellation
cancelledAt
cancelReason
cartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes cartDiscountAmountSet
channelInformation {
    displayName
    } # closes channelInformation
closed
closedAt
currentCartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentCartDiscountAmountSet
currentShippingPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentShippingPriceSet
currentSubtotalLineItemsQuantity
currentSubtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentSubtotalPriceSet
currentTaxLines {
    priceSet {
        shopMoney {
            amount
            } # closes shopMoney
        } # closes priceSet
    } # closes currentTaxLines 
currentTotalAdditionalFeesSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalAdditionalFeesSet
currentTotalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalDiscountsSet
currentTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalPriceSet
currentTotalTaxSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
discountCode
discountCodes
displayFinancialStatus
displayFulfillmentStatus
fullyPaid
originalTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
paymentGatewayNames
refunds {
    totalRefundedSet {
        shopMoney {
        amount
        } # closes shopMoney
    } # closes totalRefundedSet
    } # closes refunds
registeredSourceUrl
returnStatus
sourceName
subtotalLineItemsQuantity
subtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes subtotalPriceSet
totalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalDiscountsSet
totalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalPriceSet
transactions {
    device {
        id
        } # closes device
    } # closes transactions

} # Closing node
} # Closing edges
} # Closing orders
} # Closing query bracket
    \"""
  ) {
    bulkOperation {
      id
      status
    }
    userErrors {
      field
      message
    }
  }
}
    """


In [4]:
replaced = last_30_days.replace("{string_date}", string_date)

In [9]:
# replaced

In [ ]:
bq_30 = utils.bulk_query_request(queries.last_30_days,SHOPIFY_ACCESS_TOKEN)

In [ ]:
bulk_query_response

In [ ]:
orders = utils.poll_for_result(SHOPIFY_ACCESS_TOKEN)

In [ ]:
orders_df_complete = dffx.add_data_to_df(orders)

In [ ]:
new_df = dffx.process_data(orders_df_complete)

In [ ]:
table_id_30_days = os.getenv('ORDERS_UPDATE')

In [ ]:
gcloud.bigquery_write_table(client, new_df, table_id_30_days)

Then, create a query where you see what would be edited. Left join.

In [8]:
ORDERS_TEST = os.getenv("ORDERS")
ORDERS_UPDATE_TEST = os.getenv("ORDERS_UPDATE")

In [8]:
ORDERS_TEST = os.getenv("ORDERS")
ORDERS_UPDATE_TEST = os.getenv("ORDERS_UPDATE")


QUERY = (
    f'''
    SELECT * FROM `{ORDERS_TEST}` 
    '''
)
query_job = client.query(QUERY)  
df = query_job.to_dataframe()

/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [9]:
QUERY_UPSERT = (
    f'''
    SELECT 
  so.returnStatus as so_returnStatus,
  so.fullyPaid as so_fullyPaid,
  so.displayFinancialStatus as so_displayFinancialStatus,
  so.displayFulfillmentStatus as so_displayFulfillmentStatus,
  so.cancelReason as so_cancelReason,
  so.cancelledAt_utc as so_cancelledAt,
  so.closed as so_closed,

  sou.returnStatus as sou_returnStatus,
  sou.fullyPaid as sou_fullyPaid,
  sou.displayFinancialStatus as sou_displayFinancialStatus,
  sou.displayFulfillmentStatus as sou_displayFulfillmentStatus,
  sou.cancelReason as sou_cancelReason,
  sou.cancelledAt_utc as sou_cancelledAt,
  sou.closed as sou_closed,
  sou.closedAt as sou_closedAt,
  so.closedAt as so_closedAt

FROM {ORDERS_TEST} so
INNER JOIN {ORDERS_UPDATE_TEST} sou
  on so.id = sou.id
where 
  so.returnStatus != sou.returnStatus OR
  so.fullyPaid != sou.fullyPaid OR
  so.displayFulfillmentStatus != sou.displayFulfillmentStatus OR
  so.displayFinancialStatus != sou.displayFinancialStatus OR
  so.cancelReason != sou.cancelReason OR
  so.closed != sou.closed    

    '''
)

In [10]:
query_job = client.query(QUERY_UPSERT)  
df_join = query_job.to_dataframe()

/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [4]:
type(len("apple"))

int

In [2]:
ORDERS_TO_BE_CHANGED = queries.orders_to_modify
UPSERT_ORDERS = queries.upsert_orders

In [4]:
# UPSERT_QUERY

'\n    SELECT \n  so.returnStatus as so_returnStatus,\n  so.fullyPaid as so_fullyPaid,\n  so.displayFinancialStatus as so_displayFinancialStatus,\n  so.displayFulfillmentStatus as so_displayFulfillmentStatus,\n  so.cancelReason as so_cancelReason,\n  so.cancelledAt_utc as so_cancelledAt,\n  so.closed as so_closed,\n\n  sou.returnStatus as sou_returnStatus,\n  sou.fullyPaid as sou_fullyPaid,\n  sou.displayFinancialStatus as sou_displayFinancialStatus,\n  sou.displayFulfillmentStatus as sou_displayFulfillmentStatus,\n  sou.cancelReason as sou_cancelReason,\n  sou.cancelledAt_utc as sou_cancelledAt,\n  sou.closed as sou_closed,\n  sou.closedAt as sou_closedAt,\n  so.closedAt as so_closedAt\n\nFROM glowing-market-295819.shopify.orders so\nINNER JOIN glowing-market-295819.shopify.orders_update sou\n  on so.id = sou.id\nwhere \n  so.returnStatus != sou.returnStatus OR\n  so.fullyPaid != sou.fullyPaid OR\n  so.displayFulfillmentStatus != sou.displayFulfillmentStatus OR\n  so.displayFinancialS

In [7]:
def upsert_orders(client, query_orders_to_change, query_upsert):

    query_job_orders_to_change = client.query(query_orders_to_change)  
    query_job_orders_to_change.result()
    
    df = query_job_orders_to_change.to_dataframe()
    print(
    "{} rows to be updated".format(
        df.shape[0]
    )
    )

    if df.shape[0] > 0:
        query_job_upsert = client.query(query_upsert)  
        query_job_upsert.result()
        print('Orders updated!')
    

In [8]:
upsert_orders(client, ORDERS_TO_BE_CHANGED, UPSERT_ORDERS)

/opt/anaconda3/envs/test/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


573 rows to be updated
Orders updated!
